# Week 3 — Industrial Reporting Agent

Convertim el flux fix `search → build_prompt → llm → rag` del notebook 02 en un bucle: **pregunta → model → eina → resultat → model → resposta**. El model decideix les crides; Python controla les eines i els càlculs. El notebook 02 queda com a baseline.

Les consultes són casos de negoci representatius sobre **dades sintètiques**, no dades reals de l’ERP. Aquesta execució fa crides a l’API i té cost. Requereix `OPENAI_API_KEY` a `.env`.

## 1. Eines disponibles

| Eina | Quan es crida | Entrades | Retorn |
| --- | --- | --- | --- |
| `get_defect_summary` | Resums i totals; una crida per període per comparar | `section`, `date_from`, `date_to` | Totals, filtres, origen i semàntica |
| `get_defect_detail` | Explicar totals amb jobs, controls i línies originals | Els mateixos filtres | Controls, defectes, identificadors i metadades |
| `get_defect_pareto` | Ordenar codis de defecte segons la seva contribució | Els mateixos filtres i `metric` | Quantitats, percentatges, acumulats i referències originals |

Els filtres accepten `null` si es demana explícitament tot l’abast; les dates ISO tenen límits inclusius. Pareto admet `errors`, `affected_boards`, `repaired_boards` i `removed_boards`. La funció Python usa `removed_boards` per defecte; l’esquema estricte de l’agent exigeix els quatre arguments i el model selecciona aquesta mètrica si no se n’especifica cap.

Els percentatges expressen contribucions a la mètrica, no taxes de defectuositat ni causes. Les quantitats són ocurrències registrades, no plaques úniques. Amb total zero, els percentatges són `null` i es conserven els codis observats. Un Pareto buit no distingeix controls sans de manca de registres: cal consultar el resum.

Les taxes amb denominadors validats i l’agregació per job queden pendents. No exposem `load_defects`, SQL, calculadora genèrica ni cerca documental a l’agent.


In [2]:
from pathlib import Path
import sys
import os
import json
from dotenv import load_dotenv

repo_root = Path.cwd()
if not (repo_root / "est_prog" / "capstone_reporting").is_dir():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))
load_dotenv(repo_root / ".env")

from est_prog.capstone_reporting import get_defect_summary, get_defect_pareto, get_defect_detail
from est_prog.capstone_reporting.agent import TOOLS, run_agent
from math import isclose
MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
REFERENCE_DATE = "2026-10-16"  # Fixada perquè «mes passat» sigui reproduïble.
print("Model:", MODEL)
print("Tools:", [tool["name"] for tool in TOOLS])

Model: gpt-4o-mini
Tools: ['get_defect_summary', 'get_defect_detail', 'get_defect_pareto']


### Implementació

`est_prog/capstone_reporting/agent.py` defineix els esquemes estrictes, les instruccions i el bucle de Responses API. Conserva totes les sortides del model i retorna cada resultat amb el seu `call_id`. El dispatcher valida els arguments i només permet les tres funcions aprovades. Hi ha un límit de rondes i errors explícits si no arriba una resposta completa.

Es manté `gpt-4o-mini` del baseline, configurable amb `OPENAI_MODEL`. Les funcions de dades continuen sense dependències d’OpenAI; el capstone requereix Python 3.13 o superior.

Referència: [OpenAI — Function calling](https://developers.openai.com/api/docs/guides/function-calling).

**Migració async:** les crides actuals fan servir `await run_agent(...)`. Les sortides desades corresponen a les execucions històriques anteriors; no s’han tornat a generar amb aquesta migració.


## 2. Demostració determinista: resum → Pareto → detall

Aquesta cel·la no fa crides a l’API. Mostra la nova funció i reconcilia les quatre mètriques. Per TE de setembre de 2026, les retirades són C=8, A=3 i B=1; total=12.


In [3]:
september = dict(section="TE", date_from="2026-09-01", date_to="2026-09-30")

def reconcile_pareto(pareto, summary, detail):
    metric = pareto["metric"]
    assert pareto["source"] == summary["source"] == detail["source"]
    assert pareto["filters"] == summary["filters"] == detail["filters"]
    assert pareto["total_quantity"] == summary["totals"][metric]
    assert sum(item["quantity"] for item in pareto["items"]) == pareto["total_quantity"]
    originals = sorted((control["job_id"], control["control_id"], row["record_id"],
                        row["defect_code"], row[metric])
                       for control in detail["controls"] for row in control["defects"])
    references = []
    cumulative = 0
    for item in pareto["items"]:
        assert sum(ref["quantity"] for ref in item["records"]) == item["quantity"]
        references.extend((ref["job_id"], ref["control_id"], ref["record_id"],
                           item["defect_code"], ref["quantity"]) for ref in item["records"])
        cumulative += item["quantity"]
        if pareto["total_quantity"]:
            assert isclose(item["percentage"], 100.0 * item["quantity"] / pareto["total_quantity"])
            assert isclose(item["cumulative_percentage"], 100.0 * cumulative / pareto["total_quantity"])
        else:
            assert item["percentage"] is None and item["cumulative_percentage"] is None
    assert sorted(references) == originals  # Totes les línies, exactament una vegada.
    assert pareto["items"] == sorted(pareto["items"], key=lambda item: (-item["quantity"], item["defect_code"]))

summary = get_defect_summary(**september)
detail = get_defect_detail(**september)
for metric in ("errors", "affected_boards", "repaired_boards", "removed_boards"):
    reconcile_pareto(get_defect_pareto(metric=metric, **september), summary, detail)
pareto = get_defect_pareto(**september)
assert [(item["defect_code"], item["quantity"]) for item in pareto["items"]] == [
    ("DEMO-CODE-C", 8), ("DEMO-CODE-A", 3), ("DEMO-CODE-B", 1)]
assert pareto["total_quantity"] == 12
assert pareto["items"][-1]["cumulative_percentage"] == 100.0
print("SUMMARY", json.dumps(summary, ensure_ascii=False))
print("PARETO", json.dumps(pareto, ensure_ascii=False, indent=2))
print("DETAIL", json.dumps(detail, ensure_ascii=False))
print("PASS: four Pareto metrics reconcile with summary and every source row.")


SUMMARY {"schema_version": 1, "source": {"kind": "synthetic", "dataset_id": "synthetic-defects-v1"}, "filters": {"section": "TE", "date_from": "2026-09-01", "date_to": "2026-09-30"}, "metric_semantics": {"status": "prototype_only_not_validated_erp_rules", "date_basis": "synthetic_control_date_inclusive", "sample_quantity": "sum_once_per_control_not_unique_boards", "board_counts": "sum_of_recorded_occurrences_not_unique_boards"}, "totals": {"errors": 23, "affected_boards": 19, "repaired_boards": 4, "removed_boards": 12, "control_count": 3, "sample_quantity": 220}}
PARETO {
  "schema_version": 1,
  "source": {
    "kind": "synthetic",
    "dataset_id": "synthetic-defects-v1"
  },
  "filters": {
    "section": "TE",
    "date_from": "2026-09-01",
    "date_to": "2026-09-30"
  },
  "metric_semantics": {
    "status": "prototype_only_not_validated_erp_rules",
    "date_basis": "synthetic_control_date_inclusive",
    "sample_quantity": "sum_once_per_control_not_unique_boards",
    "board_cou

## 3. Avaluació de l’agent amb onze consultes

Les cel·les següents fan crides a l’API i mostren cada eina, els arguments, el resultat i la resposta final.


In [4]:
cases = [
    ("summary", "Dona'm el resum de defectes de la secció TE de l'1 al 30 de setembre de 2026."),
    ("detail", "Mostra el resum de TE de setembre de 2026 i els jobs i registres que expliquen les retirades."),
    ("pareto", "Mostra el Pareto de defectes de TE de setembre de 2026, amb percentatges i acumulats."),
    ("pareto_errors", "Mostra el Pareto per nombre d'errors de TE de setembre de 2026, amb percentatges i acumulats."),
    ("pareto_zero", "Mostra el Pareto de retirades de TE de l'1 al 31 d'octubre de 2026."),
    ("pareto_empty", "Mostra el Pareto de retirades i el resum de TE de gener de 2027."),
    ("comparison", "Compara els totals de defectes de TE d'agost i setembre de 2026, sense calcular taxes ni diferències."),
    ("relative", "Dona'm el resum de defectes de TE del mes passat."),
    ("empty", "Dona'm el resum de TE de gener de 2027."),
    ("clarify", "Com han anat els defectes?"),
    ("rate", "Quina és la taxa de defectes de TE de setembre de 2026?"),
]
results = {}
for name, query in cases:
    print("\n" + "=" * 60 + "\n" + name + ": " + query, flush=True)
    result = await run_agent(query, model=MODEL, today=REFERENCE_DATE)
    results[name] = result
    for call in result["tool_calls"]:
        print("TOOL", call["name"], call["arguments"])
        print(json.dumps(call["result"], ensure_ascii=False))
    print("ANSWER\n" + result["answer"], flush=True)


summary: Dona'm el resum de defectes de la secció TE de l'1 al 30 de setembre de 2026.
TOOL get_defect_summary {"section":"TE","date_from":"2026-09-01","date_to":"2026-09-30"}
{"schema_version": 1, "source": {"kind": "synthetic", "dataset_id": "synthetic-defects-v1"}, "filters": {"section": "TE", "date_from": "2026-09-01", "date_to": "2026-09-30"}, "metric_semantics": {"status": "prototype_only_not_validated_erp_rules", "date_basis": "synthetic_control_date_inclusive", "sample_quantity": "sum_once_per_control_not_unique_boards", "board_counts": "sum_of_recorded_occurrences_not_unique_boards"}, "totals": {"errors": 23, "affected_boards": 19, "repaired_boards": 4, "removed_boards": 12, "control_count": 3, "sample_quantity": 220}}
ANSWER
Dades sintètiques del resum de defectes de la secció TE de l'1 al 30 de setembre de 2026:

- **Errors:** 23
- **Plaques afectades:** 19
- **Plaques reparades:** 4
- **Plaques retirades:** 12
- **Nombre de controls:** 3
- **Mostra:** 220

Dataset ID: synt

## 4. Verificació de les crides i els fets

Els onze casos inclouen Pareto per retirades (mètrica per defecte), errors (mètrica explícita), total zero amb referències i selecció buida. Comprovem les eines, els arguments, els resultats i la reconciliació amb resum i detall.

Aquestes comprovacions no certifiquen tota la prosa: cal revisar també l’origen sintètic, els filtres ISO, el dataset_id, els buits de cobertura i l’absència de taxes o causalitat inventades. La clarificació és una resposta vàlida sense eines. Cada consulta és independent; una resposta a la clarificació s’ha d’enviar com una nova pregunta completa.


In [4]:
def successful_calls(name, tool):
    return [c for c in results[name]["tool_calls"]
            if c["name"] == tool and "error" not in c["result"]]

september = dict(section="TE", date_from="2026-09-01", date_to="2026-09-30")
for name in ("summary", "detail", "relative"):
    calls = successful_calls(name, "get_defect_summary")
    assert calls and all(json.loads(c["arguments"]) == september for c in calls)
    assert calls[0]["result"]["totals"] == dict(
        control_count=3, sample_quantity=220, errors=23,
        affected_boards=19, repaired_boards=4, removed_boards=12)

detail = successful_calls("detail", "get_defect_detail")
assert detail and detail[0]["result"]["filters"] == september
rows = [(c["job_id"], d["record_id"], d["removed_boards"])
        for c in detail[0]["result"]["controls"] for d in c["defects"]]
assert rows == [("DEMO-JOB-B", "DEMO-DEF-001", 3),
                ("DEMO-JOB-B", "DEMO-DEF-002", 1),
                ("DEMO-JOB-C", "DEMO-DEF-004", 8)]
comp = successful_calls("comparison", "get_defect_summary")
assert {c["result"]["filters"]["date_from"] for c in comp} == {"2026-08-01", "2026-09-01"}
assert all(c["result"]["filters"]["section"] == "TE" for c in comp)
assert {c["result"]["filters"]["date_to"] for c in comp} == {"2026-08-31", "2026-09-30"}
empty = successful_calls("empty", "get_defect_summary")
assert empty and empty[0]["result"]["filters"] == dict(
    section="TE", date_from="2027-01-01", date_to="2027-01-31")
assert empty[0]["result"]["totals"]["control_count"] == 0
assert results["clarify"]["tool_calls"] == []
assert all(r["answer"].strip() for r in results.values())
for name, metric, filters in (
    ("pareto", "removed_boards", september),
    ("pareto_errors", "errors", september),
    ("pareto_zero", "removed_boards", dict(section="TE", date_from="2026-10-01", date_to="2026-10-31")),
    ("pareto_empty", "removed_boards", dict(section="TE", date_from="2027-01-01", date_to="2027-01-31")),
):
    calls = successful_calls(name, "get_defect_pareto")
    assert calls, (name, "Missing Pareto call")
    expected = get_defect_pareto(metric=metric, **filters)
    for call in calls:
        assert json.loads(call["arguments"]) == dict(filters, metric=metric)
        assert call["result"] == expected
        reconcile_pareto(call["result"], get_defect_summary(**filters), get_defect_detail(**filters))
empty_summary = successful_calls("pareto_empty", "get_defect_summary")
assert empty_summary and all(c["result"]["totals"]["control_count"] == 0 for c in empty_summary)
assert all(c["result"]["filters"] == dict(section="TE", date_from="2027-01-01", date_to="2027-01-31") for c in empty_summary)
assert len(results) == 11
print("PASS: Pareto routing, metrics, rankings, percentages, zero/empty results and reconciliation.")
print("PASS: tools, filters, totals, source rows, relative month, empty selection and clarification.")
print("Manual review: read the answers above, especially the unsupported rate request.")

PASS: Pareto routing, metrics, rankings, percentages, zero/empty results and reconciliation.
PASS: tools, filters, totals, source rows, relative month, empty selection and clarification.
Manual review: read the answers above, especially the unsupported rate request.


## 5. Límits i següent pas

Les tres eines inclouen resum, Pareto i detall. Aquest increment demostra l’orquestració sobre un fixture fix. No hi ha connexió ERP, garanties de cobertura temporal, definició de taxa, agregació per job ni memòria entre consultes. Les instruccions guien la prosa, però no en garanteixen la fidelitat: les respostes i les traces queden visibles per revisar-les. Abans de connectar dades reals cal validar les regles de negoci i ampliar l’avaluació amb informes i resultats de referència.

## Revisió de l’execució guardada (18 d’octubre de 2026)

Execució amb Python 3.13.16 i `gpt-4o-mini`. La demostració directa reconcilia les quatre mètriques amb resum i detall. Els onze casos de l’agent passen les comprovacions de crides, arguments i resultats, inclosos el Pareto per retirades, el Pareto per errors, el total zero i la selecció buida. Les traces i respostes d’aquesta execució es conserven a les cel·les anteriors.

**Nova funció validada:** per TE de setembre, les retirades són C=8, A=3 i B=1; total=12; acumulats 66,67%, 91,67% i 100%. Per errors, C=10, A=8 i B=5; total=23. Els percentatges es calculen a l’eina sense arrodoniment de presentació i totes les línies originals es reconcilien exactament una vegada.

**Límits observats en la resposta del model:**

- `pareto_zero`: la tool conserva `DEMO-DEF-006` amb quantitat zero i percentatges `null`. La resposta enumera aquest registre, però afegeix incorrectament «sense registres coincidents; cobertura desconeguda» i omet el dataset_id i els percentatges indefinits. Hi ha un control coincident; no és una selecció buida.
- `detail`: presenta 3 retirades sota `DEMO-CTRL-002`, tot i que les seves dues línies sumen 4 (3+1). El detall retornat per l’eina i el total global de 12 són correctes.
- `comparison`: omet l’avís de cobertura desconeguda per a l’agost, que no té controls coincidents.
- `rate`: no inventa una taxa numèrica, però afirma sense base que la cobertura és vàlida i no explica que no hi ha denominador validat; també omet el dataset_id.
- Algunes respostes ometen els límits ISO explícits o l’avís que els percentatges de Pareto són contribucions, no taxes.

Per tant, **PASS valida les crides i els resultats de les eines; no certifica tota la prosa del model**. La sortida determinista i les traces són l’evidència per a l’entrega. El següent pas és validar estructuradament la resposta i presentar filtres, semàntica i cobertura des de Python.

La data de referència continua fixada al 16 d’octubre de 2026 per reproduir «mes passat». Les dades són sintètiques; aquest exercici no valida regles ERP ni causes de negoci.
